# Club Convergence Analysis

This notebook applies the **Phillips–Sul club convergence methodology** to panel data.

## Workflow

1. Load per-capita data in wide format.
2. Log-transform the data.
3. Remove short-run fluctuations with the Hodrick–Prescott (HP) filter.
4. Test for overall convergence using the log-t test.
5. Identify convergence clubs.
6. Merge clubs using the Phillips–Sul procedure.
7. Inspect summaries and transition-path plots.

## Expected input

The input CSV should have:

- **one row per country/region**;
- **one identifier column first** (for example, `Country`);
- **one numeric column per time period**;
- no missing values;
- strictly positive values, because the analysis uses logarithms.

The notebook keeps only the two packages actually used directly:

- `ConvergenceClubs` — Phillips–Sul convergence tests and club identification.
- `mFilter` — Hodrick–Prescott filtering.

The other packages from the original notebook (`tidyverse`, `tibble`, `data.table`, `plyr`, `dplyr`, `janitor`, `oglmx`, and `rms`) were not used by the analysis and have been removed.


## 1. Load the required packages

For a GitHub repository, package installation is intentionally kept separate from the analysis. Install the two packages once in your R environment if they are not already available.

```r
install.packages(c("ConvergenceClubs", "mFilter"))
```


In [ ]:
# Load the package used for Phillips–Sul convergence tests and club identification.
library(ConvergenceClubs)

# Load the package that provides the Hodrick–Prescott filter.
library(mFilter)


## 2. Analysis settings

Keep the main settings in one place so they are easy to review or change.

For annual data, a common HP-filter smoothing parameter is **100**. The `time_trim` value of `1/3` follows the common Phillips–Sul implementation used in the original notebook.


In [ ]:
# Set the path to the input CSV file.
data_path <- "mf percapita.csv"

# Set the HP-filter smoothing parameter for annual data.
hp_lambda <- 100

# Trim the initial third of the sample in the Phillips–Sul log-t test.
time_trim <- 1 / 3

# Use the fixed quadratic spectral HAC estimator used in the original analysis.
hac_method <- "FQSB"

# Use the original fixed critical value setting for club identification.
cstar <- 0


## 3. Load and validate the data

The first column is treated as the country/region identifier. All remaining columns must contain numeric observations for the time periods.


In [ ]:
# Read the panel data from the CSV file.
mf <- read.csv(
  file = data_path,
  check.names = FALSE,
  stringsAsFactors = FALSE
)

# Stop if the dataset does not contain an identifier column and at least two time periods.
if (ncol(mf) < 3) {
  stop("The dataset must contain an identifier column and at least two time-period columns.")
}

# Stop if the identifier column contains missing values.
if (anyNA(mf[[1]])) {
  stop("The first column contains missing country/region identifiers.")
}

# Check that every time-period column is numeric.
if (!all(vapply(mf[-1], is.numeric, logical(1)))) {
  stop("All columns after the first identifier column must be numeric.")
}

# Stop if any missing or non-finite values are present in the numeric panel.
if (any(!is.finite(as.matrix(mf[-1])))) {
  stop("The numeric panel contains missing, infinite, or non-finite values.")
}

# Stop if any value is zero or negative because logarithms require positive values.
if (any(as.matrix(mf[-1]) <= 0)) {
  stop("All numeric observations must be strictly positive before log transformation.")
}

# Display the first rows so the input structure can be checked.
head(mf)


## 4. Log-transform the panel

The Phillips–Sul procedure is applied to the logged per-capita series. The identifier column is kept unchanged.


In [ ]:
# Extract the numeric time-series panel and take natural logarithms.
logmf <- log(mf[-1])

# Keep the original country/region identifiers for later reconstruction of the filtered panel.
country_names <- mf[[1]]


## 5. Remove short-run fluctuations with the HP filter

The HP filter is applied independently to each country/region.

The filtered **trend component** is retained for the convergence analysis. For annual data, `hp_lambda = 100` is used here, matching the original notebook.

Typical values are:

- annual data: `100`
- quarterly data: `1600`
- monthly data: `14400`

These values should be reviewed for the frequency and research design of the dataset.


In [ ]:
# Apply the HP filter to each country's logged time series.
filtered_values <- t(
  apply(
    X = logmf,
    MARGIN = 1,
    FUN = function(x) {
      hpfilter(
        x = x,
        freq = hp_lambda,
        type = "lambda"
      )$trend
    }
  )
)

# Convert the filtered matrix back to a data frame and restore the original column names.
flogmf <- data.frame(
  country_names,
  filtered_values,
  check.names = FALSE,
  stringsAsFactors = FALSE
)

# Give the first column the same name as the original identifier column.
names(flogmf)[1] <- names(mf)[1]

# Verify the filtered panel before running the convergence procedure.
if (any(!is.finite(as.matrix(flogmf[-1])))) {
  stop("The HP-filtered panel contains non-finite values.")
}

# Display the first rows of the filtered panel.
head(flogmf)


## 6. Phillips–Sul log-t test

First, calculate the cross-sectional variance measure `H`.

The log-t test evaluates the null hypothesis of overall convergence. If overall convergence is rejected, the club-identification procedure is used to determine groups of units that converge to different steady states.


In [ ]:
# Compute the cross-sectional variance sequence used by the Phillips–Sul log-t test.
H <- computeH(
  flogmf[-1],
  quantity = "H"
)

# Estimate the log-t model using the selected trimming and HAC settings.
overall_test <- estimateMod(
  H,
  time_trim = time_trim,
  HACmethod = hac_method
)

# Display the estimated coefficient and test statistics rounded to three decimals.
round(overall_test, 3)


### Interpretation

The null hypothesis is **overall convergence**.

If the log-t test rejects overall convergence, this does not imply that no convergence exists. It motivates the next step: identifying **convergence clubs**, where subsets of countries/regions may converge to different long-run paths.


## 7. Identify convergence clubs

The `ConvergenceClubs` package expects units in rows and time periods in columns.

The column positions are generated dynamically rather than hard-coded, which makes the notebook easier to reuse with datasets containing a different number of years.


In [ ]:
# Identify the columns containing the time-series observations.
data_columns <- 2:ncol(flogmf)

# Use the final time period as the reference column for ordering the units.
reference_column <- ncol(flogmf)

# Run the Phillips–Sul club-identification procedure.
clubs <- findClubs(
  flogmf,
  dataCols = data_columns,
  unit_names = 1,
  refCol = reference_column,
  time_trim = time_trim,
  cstar = cstar,
  HACmethod = hac_method
)

# Display a compact summary of the identified clubs.
summary(clubs)


In [ ]:
# Print the detailed convergence-club results, including club membership and test information.
print(clubs)


## 8. Plot the initial convergence clubs

The first plot shows the transition paths for the identified clubs.


In [ ]:
# Plot the transition paths for the identified convergence clubs.
plot(clubs)


In [ ]:
# Plot the average transition path for each club and include the legend.
plot(
  clubs,
  clubs = NULL,
  avgTP = TRUE,
  legend = TRUE
)


## 9. Merge convergence clubs

The Phillips–Sul merging procedure can combine clubs that satisfy the relevant convergence criteria.

`mergeMethod = "PS"` follows the Phillips–Sul merging approach used in the original notebook.


In [ ]:
# Merge compatible convergence clubs using the Phillips–Sul procedure.
mclubs <- mergeClubs(
  clubs,
  mergeMethod = "PS"
)

# Display a summary of the merged clubs.
summary(mclubs)


In [ ]:
# Print the detailed results after club merging.
print(mclubs)


## 10. Plot the merged clubs

These plots show the convergence structure after the merging step.


In [ ]:
# Plot the transition paths for the merged convergence clubs.
plot(mclubs)


In [ ]:
# Plot the average transition path for each merged club and include the legend.
plot(
  mclubs,
  clubs = NULL,
  avgTP = TRUE,
  legend = TRUE
)


## 11. Notes for reproducibility

Before committing this notebook to GitHub:

- Keep the input CSV in a documented location or add it to the repository if its license permits redistribution.
- Do not commit sensitive or proprietary data.
- Record the R version and package versions used for the final results.
- If the analysis is part of a paper or report, document why the HP-filter parameter and Phillips–Sul settings were chosen.
- Consider adding a `renv` lockfile later if you need exact package-version reproducibility.
